<div dir='rtl'>

# 🎬 خادم الدبلجة العربية بأصوات الممثلين — الإصدار 2

يعمل مجانًا على بطاقة رسومية من Google، ويُدار من تطبيق **مدير التنزيلات** على هاتفك.
يعزل أصوات الممثلين عن الموسيقى، ويفهم الكلام، ويميّز كل ممثل، ويترجم بدقة، ثم ينطق كل جملة **بصوت ممثلها ونبرته وانفعاله** وبطول الجملة الأصلية، فوق الموسيقى والمؤثرات الأصلية.

1. اضغط زر التشغيل ▶️ في الخلية التالية (وإن ظهر تحذير اضغط **Run anyway**).
2. انتظر حتى تظهر عبارة **الخادم جاهز** ثم اضغط **نسخ الرابط**.
3. ارجع إلى التطبيق ← صفحة **الدبلجة** ← خيار **أصوات الممثلين (مجاني – Colab)**.

</div>

In [ ]:
#@title ▶️ اضغط زر التشغيل ◀ لتشغيل خادم الدبلجة (انتظر 3–5 دقائق)
import os, re, sys, time, glob, html, site, secrets, subprocess, threading
from IPython.display import display, HTML

def msg(t, big=False):
    display(HTML(f"<div dir='rtl' style='font-size:{22 if big else 17}px;margin:6px 0'>{t}</div>"))

try:
    import torch
    gpu = torch.cuda.is_available()
except Exception:
    gpu = False
if not gpu:
    msg("⚠️ لا توجد بطاقة رسومية. من القائمة: <b>Runtime ← Change runtime type ← T4 GPU ← Save</b> ثم اضغط زر التشغيل مرة أخرى.", True)
    raise SystemExit

msg("⏳ 1/3 تثبيت أدوات الدبلجة (مرة واحدة في كل جلسة، 3–5 دقائق)…")
pkgs = ["omnivoice", "transformers", "accelerate", "bitsandbytes", "faster-whisper", "audio-separator[cpu]",
        "speechbrain", "num2words", "scikit-learn", "librosa", "soundfile", "sentencepiece", "huggingface_hub"]
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U"] + pkgs, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-3000:], r.stderr[-3000:])

msg("⏳ 2/3 تشغيل الخادم وإنشاء الرابط…")
subprocess.run(["wget", "-q", "-O", "/content/cloudflared",
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"])
os.chmod("/content/cloudflared", 0o755)
subprocess.run(["wget", "-q", "-O", "/content/dub_server.py",
                f"https://raw.githubusercontent.com/yaellah27625703-eng/downloader/main/colab/dub_server.py?t={int(time.time())}"])
token = secrets.token_urlsafe(16)
# the speech-to-text engine needs the GPU libraries that came with PyTorch
libs = [d for sp in site.getsitepackages() for d in glob.glob(os.path.join(sp, "nvidia", "*", "lib"))]
env = dict(os.environ, LD_LIBRARY_PATH=":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")]),
           HF_HUB_DISABLE_PROGRESS_BARS="1", TOKENIZERS_PARALLELISM="false")
log = open("/content/server.log", "w")
srv = subprocess.Popen([sys.executable, "-u", "/content/dub_server.py", "--port", "8765", "--token", token],
                       stdout=log, stderr=subprocess.STDOUT, env=env)
tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8765"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url, t0 = None, time.time()
for line in tun.stdout:
    m = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
    if time.time() - t0 > 180:
        break
threading.Thread(target=lambda: [None for _ in tun.stdout], daemon=True).start()
if not url:
    msg("❌ تعذّر إنشاء الرابط. اضغط زر التشغيل مرة أخرى.", True)
    raise SystemExit

link = f"{url}/?k={token}"
display(HTML(f"""
<div dir='rtl' style='font-size:18px;line-height:1.7'>
<h2>✅ 3/3 الخادم جاهز</h2>
<p>1) اضغط <b>نسخ الرابط</b> (أو اضغط مطوّلًا على الرابط وانسخه).<br>
2) ارجع إلى تطبيق <b>مدير التنزيلات</b> ← صفحة <b>الدبلجة</b>، وسيتصل تلقائيًا.<br>
3) <b>اترك هذه الصفحة مفتوحة</b> طوال الدبلجة. تُنزَّل النماذج الآن في الخلفية (أول مرة 5–10 دقائق).</p>
<input id='lnk' value='{link}' readonly style='width:100%;font-size:16px;padding:8px' onclick='this.select()'>
<button style='font-size:22px;padding:12px 24px;margin-top:10px' onclick="navigator.clipboard.writeText('{link}').then(()=>{{this.innerText='✓ نُسخ الرابط'}}).catch(()=>{{document.getElementById('lnk').select();document.execCommand('copy');this.innerText='✓ نُسخ'}})">📋 نسخ الرابط</button>
</div>"""))

# keep this cell running: it keeps the session (and the free GPU) alive, and shows what the server is doing
status = display(HTML("<div dir='rtl' style='font-size:16px;color:#666'>📋 الحالة: يبدأ الخادم…</div>"), display_id=True)
last = ""
while True:
    time.sleep(15)
    if srv.poll() is not None:
        msg("❌ توقف الخادم. اضغط زر التشغيل مرة أخرى.", True)
        print(open("/content/server.log", encoding="utf-8", errors="ignore").read()[-3000:])
        break
    try:
        lines = [l for l in open("/content/server.log", encoding="utf-8", errors="ignore").read().splitlines()
                 if re.match(r"\d\d:\d\d:\d\d ", l)]
        if lines and lines[-1] != last:
            last = lines[-1]
            status.update(HTML(f"<div dir='rtl' style='font-size:16px;color:#666'>📋 الحالة: {html.escape(last)}</div>"))
    except Exception:
        pass
